In [0]:
from pyspark.sql import functions as F
from py4j.protocol import Py4JJavaError


RAW_BASE_PATH = "s3://ifood-case-data-quality/raw-data"
LANDING_BASE_PATH = "s3://ifood-case-data-quality/landing-data"


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as error:
        message = str(error)

        if (
            "java.io.FileNotFoundException" in message
            or "Path does not exist" in message
            or "No such file or directory" in message
        ):
            return False

        raise


def process_taxi_service_month(
    service_type: str,
    year: int | str,
    month: int | str,
) -> bool:
    year_str = str(year)
    month_str = f"{int(month):02d}"

    taxi_folder = f"{service_type}_taxi"

    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{taxi_folder}/"
        f"year={year_str}/"
        f"month={month_str}/"
    )

    landing_path = f"{LANDING_BASE_PATH}/{taxi_folder}/"

    if not path_exists(raw_path):
        print(f"Path não encontrado, ignorando: {raw_path}")
        return False

    print(f"Lendo: {raw_path}")

    df = spark.read.parquet(raw_path)

    df = (
        df
        .withColumn("id_data", F.lit(f"{service_type}_{year_str}_{month_str}"))
        .withColumn("year", F.lit(year_str))
        .withColumn("month", F.lit(month_str))
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("year", "month")
        .save(landing_path)
    )

    print(f"Gravado com sucesso em: {landing_path}")
    return True

In [0]:
from datetime import date

service_types = ["yellow", "green", "fhv", "fhvhv"]
year = 2026

atual_month = date.today().month
months = range(1, atual_month)

for service_type in service_types:
    for month in months:
        print(month)
        process_taxi_service_month(service_type, year, month)

1
Lendo: s3://ifood-case-data-quality/raw-data/yellow_taxi/year=2026/month=01/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/yellow_taxi/
2
Lendo: s3://ifood-case-data-quality/raw-data/yellow_taxi/year=2026/month=02/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/yellow_taxi/
3
Lendo: s3://ifood-case-data-quality/raw-data/yellow_taxi/year=2026/month=03/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/yellow_taxi/
4
Lendo: s3://ifood-case-data-quality/raw-data/yellow_taxi/year=2026/month=04/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/yellow_taxi/
1
Lendo: s3://ifood-case-data-quality/raw-data/green_taxi/year=2026/month=01/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/green_taxi/
2
Lendo: s3://ifood-case-data-quality/raw-data/green_taxi/year=2026/month=02/
Gravado com sucesso em: s3://ifood-case-data-quality/landing-data/green_taxi/
3
Lendo: s3://ifood-case-data-quality/raw-data/green_tax